In [ ]:
from model import Registrater
from dataset_pre import Data_Loader
from torch.utils.data import Subset
import sys
import os

root_dir = os.path.abspath(os.path.join(os.path.join(os.getcwd(), "../.."), "../.."))
sys.path.append(root_dir)
from trainer import BaseTrainModule, Trainer

import torch
from torchvision import transforms
import torch.nn.functional as F
from torch.utils.data import Dataset, random_split
import matplotlib.pyplot as plt
from torch import optim
import torch.nn as nn
import os

ROOT_PATH = os.path.abspath(os.path.join(os.path.join(os.path.join(os.getcwd(), os.pardir), os.pardir), os.pardir))



def show_layers(layers, title=''):
    if isinstance(layers, torch.Tensor):
        layers = layers.cpu().detach().numpy()[0]

    c, _, _ = layers.shape

    plt.figure(figsize=(c, 1.5))
    for i in range(c):
        plt.subplot(1, c, (i+1))
        plt.imshow(layers[i], 'gray', vmax=1, vmin=0)
        plt.axis('off')
        plt.title('{} L{}'.format(title, c))
    plt.show()

class MyTrainModule(BaseTrainModule):
    def __init__(self, reduce):
        super().__init__()
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        self.reduce = reduce

        self.model = Registrater(in_channels=4)
        self.model = nn.DataParallel(self.model)
        self.model.to(device=self.device)


    def configure_lossfunctions(self):
        self.criterion = nn.MSELoss()

    def configure_optimizers(self, lr):
        self.optimizer = optim.Adam(self.model.parameters(), lr=lr, betas=(0.5, 0.999))

        return self.optimizer

    def configure_scheduler(self, optimizers):
        optimizer = optimizers
        self.scheduler = torch.optim.lr_scheduler.StepLR(optimizer=optimizer, step_size=50, gamma=0.5)

        return (self.scheduler)

    def configure_logs(self):
        # learning rate
        self.set_log(name='Lr', obj=self.optimizer, category='lr')
        # loss function
        self.set_log(name='Loss_reg', obj=self.criterion, category='loss', mode='train')
        self.set_log(name='Loss_reg_val', obj=self.criterion, category='loss', mode='valid')

    def training_step(self, batch_idx, batch):
        self.model.train()
        moving_image, fixed_image, moving_masks, fixed_masks, parmeter_gt = batch
        
        self.optimizer.zero_grad()
        warped, fixed, warped_mask, fixed_mask, parmeter = self.model(moving_image, fixed_image, moving_masks, fixed_masks)

        # print('\t GT upper: S({:.5f}) R({:.5f}) T({:.5f}, {:.5f})'.format(parmeter_gt.cpu().detach().numpy()[0][0][0], parmeter_gt.cpu().detach().numpy()[0][0][1],
        #                                                                    parmeter_gt.cpu().detach().numpy()[0][0][2], parmeter_gt.cpu().detach().numpy()[0][0][3]),
        # ' GT lower: S({:.5f}) R({:.5f}) T({:.5f}, {:.5f})'.format(parmeter_gt.cpu().detach().numpy()[0][1][0], parmeter_gt.cpu().detach().numpy()[0][1][1],
        #                                                                    parmeter_gt.cpu().detach().numpy()[0][1][2], parmeter_gt.cpu().detach().numpy()[0][1][3]))
        
        # print('\t Pr upper: S({:.5f}) R({:.5f}) T({:.5f}, {:.5f})'.format(parmeter.cpu().detach().numpy()[0][0][0], parmeter.cpu().detach().numpy()[0][0][1],
        #                                                                    parmeter.cpu().detach().numpy()[0][0][2], parmeter.cpu().detach().numpy()[0][0][3]),
        # ' Pr lower: S({:.5f}) R({:.5f}) T({:.5f}, {:.5f})'.format(parmeter.cpu().detach().numpy()[0][1][0], parmeter.cpu().detach().numpy()[0][1][1],
        #                                                                    parmeter.cpu().detach().numpy()[0][1][2], parmeter.cpu().detach().numpy()[0][1][3]))

        loss_I = torch.sqrt(self.criterion(warped, fixed))
        loss_P = torch.sqrt(self.criterion(parmeter, parmeter_gt))
        loss = 0.5 * loss_I + 0.5 *loss_P
        print(self.reduce, 'Image_loss: ', loss_I.item(), 'Parameter_loss: ', loss_P.item())

        loss.backward()
        self.optimizer.step()

        return loss

    def validation_step(self, batch_idx, batch):
        self.model.eval()
        moving_image, fixed_image, moving_masks, fixed_masks, parmeter_gt = batch

        warped, fixed, warped_mask, fixed_mask, parmeter = self.model(moving_image, fixed_image, moving_masks, fixed_masks)

        # print('\t GT upper: S({:.5f}) R({:.5f}) T({:.5f}, {:.5f})'.format(parmeter_gt.cpu().detach().numpy()[0][0][0], parmeter_gt.cpu().detach().numpy()[0][0][1],
        #                                                                    parmeter_gt.cpu().detach().numpy()[0][0][2], parmeter_gt.cpu().detach().numpy()[0][0][3]),
        # ' GT lower: S({:.5f}) R({:.5f}) T({:.5f}, {:.5f})'.format(parmeter_gt.cpu().detach().numpy()[0][1][0], parmeter_gt.cpu().detach().numpy()[0][1][1],
        #                                                                    parmeter_gt.cpu().detach().numpy()[0][1][2], parmeter_gt.cpu().detach().numpy()[0][1][3]))
        
        
        # print('\t Pr upper: S({:.5f}) R({:.5f}) T({:.5f}, {:.5f})'.format(parmeter.cpu().detach().numpy()[0][0][0], parmeter.cpu().detach().numpy()[0][0][1],
        #                                                                    parmeter.cpu().detach().numpy()[0][0][2], parmeter.cpu().detach().numpy()[0][0][3]),
        # ' Pr lower: S({:.5f}) R({:.5f}) T({:.5f}, {:.5f})'.format(parmeter.cpu().detach().numpy()[0][1][0], parmeter.cpu().detach().numpy()[0][1][1],
        #                                                                    parmeter.cpu().detach().numpy()[0][1][2], parmeter.cpu().detach().numpy()[0][1][3]))


        loss_I = torch.sqrt(self.criterion(warped, fixed))
        loss_P = torch.sqrt(self.criterion(parmeter, parmeter_gt))
        loss_val = 0.5 * loss_I + 0.5 *loss_P
        print('Image_loss: ', loss_I.item(), 'Parameter_loss: ', loss_P.item())

        loss_show = F.mse_loss(warped, fixed, reduction='none')

        image_list = [moving_image, fixed_image, warped, loss_show]

        return loss_val, image_list

    def configure_saveprocess(self):
        self.set_save_parameter(model=self.model, loss_name='Loss_reg',
                                save_path=ROOT_PATH + '/experiments/Exp_downstream/JSN_evaluation/parameter/best_reg_model_pre_{}.pth'.format(self.reduce))

    def show_single_log_image(self, image_box):
        moving, fixed, warped, loss_show = image_box
        plt.figure(figsize=(8, 2))
        plt.subplot(1, 4, 1)
        plt.imshow(moving[0], 'gray')
        plt.subplot(1, 4, 2)
        plt.imshow(fixed[0], 'gray')
        plt.subplot(1, 4, 3)
        plt.imshow(warped[0], 'gray')
        plt.imshow(warped[1], 'gray', alpha=0.5)
        plt.subplot(1, 4, 4)
        plt.imshow(loss_show[0], alpha=0.5)
        plt.imshow(loss_show[1], alpha=0.5)
        plt.show()



if __name__ == "__main__":
    # Data Loading
    image_size = 256
    transform = transforms.Compose([transforms.Resize((image_size, image_size)),
                                    transforms.ToTensor(),
                                    transforms.Normalize(0, 1)
                                    ])
    
    reduce_list = [10, 5, 3, 1]
    for REDUCE in reduce_list:
        print(REDUCE)
        dataset = Data_Loader(ROOT_PATH + '/experiments/Exp_downstream/JSN_evaluation/pre_dataset_{}.json'.format(REDUCE), transform)
        train_size = int(0.8 * len(dataset)) 
        test_size = len(dataset) - train_size  
        train_dataset, valid_dataset = random_split(dataset, [train_size, test_size])


        mytrainmodule = MyTrainModule(reduce=REDUCE)
        trainer = Trainer(train_module=mytrainmodule, train_dataset=train_dataset, valid_dataset=valid_dataset)
        trainer.configure(batch_size=48, epochs=100, device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
                        result_number=1, lr=1e-4)
        trainer.fit()

